In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.titleweight'] = 'bold'

DATA = 'data'

def to_dt(ms):
    return pd.to_datetime(pd.to_numeric(ms, errors='coerce'), unit='ms')

In [ ]:
labels = pd.read_csv(f'{DATA}/incident_label.csv')
labels['ts_start_dt'] = to_dt(labels['ts_start'])
labels['ts_end_dt'] = to_dt(labels['ts_end'])
labels['ts_end_f'] = labels['ts_end'].fillna(labels['ts_start'] + 3_600_000)
labels[['id', 'tier', 'kind', 'severity', 'provenance', 'ts_start_dt']].sort_values('ts_start_dt')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
colors = {1: '#d62728', 2: '#ff7f0e', 3: '#7f7f7f'}
for tier, grp in labels.groupby('tier'):
    is_lt = grp['kind'] == 'load_test'
    ax.scatter(grp.loc[~is_lt, 'ts_start_dt'], [tier]*(~is_lt).sum(),
               color=colors[tier], s=80, label=f'tier {tier}', zorder=3)
    ax.scatter(grp.loc[is_lt, 'ts_start_dt'], [tier]*is_lt.sum(),
               color=colors[tier], s=80, marker='x', zorder=3)
ax.scatter([], [], color='gray', marker='x', s=80, label='load_test (não-incidente)')
ax.set_yticks([1, 2, 3]); ax.set_ylim(0.5, 3.5)
ax.set_ylabel('tier'); ax.set_title(f'Os {len(labels)} rótulos do golden set, no tempo (X = teste de carga, não incidente)')
ax.legend(loc='upper left', fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%b'))
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

print(f"Total: {len(labels)} linhas | {(labels['kind']!='load_test').sum()} incidentes reais | "
      f"{(labels['kind']=='load_test').sum()} load_test (excluir)")
print("\nDistribuição por provenance (confiabilidade do rótulo):")
print(labels['provenance'].value_counts())

In [ ]:
inc = pd.read_csv(f'{DATA}/incidents.csv')
inc['opened_dt'] = to_dt(inc['opened_at'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
inc['priority'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='#4C72B0')
axes[0].set_title('Distribuição de prioridade (783 chamados brutos)')
axes[0].set_xlabel('priority'); axes[0].set_ylabel('qtde')

video_mask = inc['business_service'].fillna('').str.contains('Globoplay|[Vv]ídeo', regex=True)
relevant = inc[inc['priority'].isin(['P1', 'P2', 'P3']) & video_mask]
inc['relevante'] = 'ruído (P4/P5 ou não-vídeo)'
inc.loc[inc['priority'].isin(['P1','P2','P3']) & video_mask, 'relevante'] = 'P1-P3 + vídeo/Globoplay'
inc['relevante'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.0f%%', ylabel='')
axes[1].set_title('Quanto do chamado bruto é sinal de verdade')
plt.tight_layout(); plt.show()

print(f"P1-P3 + vídeo/Globoplay: {len(relevant)} chamados")
already = set(labels['id'].str.replace('itsm-', '', regex=False))
novos = sorted(set(relevant['sys_id']) - set())  # sys_id != number; comparar por número
novos_num = sorted(set(relevant['number']) - already)
print(f"Já estão no golden set: {len(set(relevant['number']) & already)}")
print(f"Ainda não rotulados (dá pra adicionar de graça, olhando o ticket): {len(novos_num)} -> {novos_num}")